# 🛡️ Multi-Domain Network Attacks & World Model Dataset Visualizer

Interactive visualization suite for the **Unified Multi-Domain Cyber Attack Dataset** (DARPA 1998 + CIC-IDS2017) and the **47-Dimensional Canonical State Vector ($)**.

---

### Dataset Highlights:
- **Total Records:** 1,545,739 state records across 13 distinct network sessions
- **Feature Space:** 47 continuous telemetry features per 10-second rolling window
- **Target Taxonomy:** Full 5-Stage MITRE ATT&CK Kill-Chain Mapping:
  - **Stage 0:** Benign Baseline
  - **Stage 1:** Reconnaissance (PortScan, IP Sweep)
  - **Stage 2:** Initial Access (SSH/FTP Brute-Force, Web Exploits)
  - **Stage 3:** Lateral Movement & Execution (Buffer Overflows, SQLi, XSS, Infiltration)
  - **Stage 4:** Denial of Service (SYN Flood, UDP/ICMP Flood, DoS Hulk, Slowloris, DDoS)


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

print('Environment initialized successfully!')


## 1. Load the Multi-Domain Dataset
Loading from all 13 session parquets (data/processed/ + data/processed_cic/) with universal cross-platform compatibility:


In [ ]:
# Load from all 13 session parquets
session_files = sorted(glob.glob('data/processed/windows_*_dt10s.parquet')) + sorted(glob.glob('data/processed_cic/*.parquet'))

dfs = []
for f in session_files:
    try:
        dfs.append(pd.read_parquet(f))
    except Exception:
        try:
            import pyarrow.parquet as pq
            dfs.append(pq.read_table(f).to_pandas())
        except Exception:
            dfs.append(pd.read_parquet(f, engine='fastparquet'))

df = pd.concat(dfs, ignore_index=True)
print(f'Master Dataset Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

# Discover feature columns
META_COLS = ['day', 'split', 'window_idx', 'window_start_time', 'window_end_time', 'active_session_count', 'is_attack', 'mitre_stage_code', 'mitre_stage_name', 'active_attack_names', 'session', 'dataset', 'raw_attack_type']
feature_cols = [c for c in df.columns if c not in META_COLS and np.issubdtype(df[c].dtype, np.number) and not c.startswith('delta_')]

print(f'Total Continuous Feature Dimensions: {len(feature_cols)}')


## 2. MITRE ATT&CK Stage Class Distribution
Visualizing sample representation across all 5 kill-chain stages:


In [ ]:
stage_counts = df['mitre_stage_name'].value_counts()
colors = ['#2b5c8f', '#d95f02', '#7570b3', '#e7298a', '#e6ab02']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
sns.barplot(x=stage_counts.values, y=stage_counts.index, palette='crest', ax=ax1)
ax1.set_title('Record Count per MITRE Attack Stage (Log Scale)', fontsize=14, fontweight='bold')
ax1.set_xscale('log')
ax1.set_xlabel('Sample Count (Log Scale)')
for i, v in enumerate(stage_counts.values):
    ax1.text(v * 1.1, i, f'{v:,}', va='center', fontweight='semibold')

# Pie plot
ax2.pie(stage_counts.values, labels=stage_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
ax2.set_title('Stage Proportion Breakdown (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


## 3. Telemetry Signature Comparison Across Stages
Visualizing the physical signatures characterizing each attack stage:
- **Packet Rate:** Extreme transmission bursts during Stage 1 (PortScan: ~52,000 pkts/s) and Stage 4 (DoS: ~26,000 pkts/s)
- **Authentication Ratio:** Collapses to 100% during Stage 2 (Initial Access / SSH/FTP Brute Force)
- **Payload Mean & Max:** Peak during Stage 4 (DoS floods) and Stage 3 (CVE Exploit Delivery)
- **Advertised TCP Window Size:** Sharp contraction during DoS attacks (connection starvation)


In [ ]:
# Compute mean telemetry values across the 5 MITRE stages
stage_means = df.groupby('mitre_stage_code')[['packet_rate', 'auth_packet_ratio', 'payload_mean', 'win_mean']].mean()

stage_labels = [
    '0: Benign',
    '1: Recon (PortScan)',
    '2: Access (BruteForce)',
    '3: Lateral (Exploits)',
    '4: DoS (Floods)'
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Packet Rate
ax1 = axes[0, 0]
sns.barplot(x=stage_labels, y=stage_means['packet_rate'], palette='Blues_r', ax=ax1)
ax1.set_title('Mean Packet Rate (pkts/sec) per Stage', fontsize=13, fontweight='bold')
ax1.set_ylabel('Packets / Second')
ax1.tick_params(axis='x', rotation=15)
for i, v in enumerate(stage_means['packet_rate']):
    ax1.text(i, v + 1000, f'{v:,.0f}', ha='center', fontweight='bold')

# 2. Authentication Ratio
ax2 = axes[0, 1]
sns.barplot(x=stage_labels, y=stage_means['auth_packet_ratio'], palette='Oranges_r', ax=ax2)
ax2.set_title('Mean Authentication Port Ratio (Ports 21/22/23)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Auth Packet Ratio (0.0 to 1.0)')
ax2.tick_params(axis='x', rotation=15)
for i, v in enumerate(stage_means['auth_packet_ratio']):
    ax2.text(i, v + 0.03, f'{v*100:.1f}%', ha='center', fontweight='bold')

# 3. Payload Mean
ax3 = axes[1, 0]
sns.barplot(x=stage_labels, y=stage_means['payload_mean'], palette='Greens_r', ax=ax3)
ax3.set_title('Mean Application Payload Size (Bytes)', fontsize=13, fontweight='bold')
ax3.set_ylabel('Payload (Bytes)')
ax3.tick_params(axis='x', rotation=15)
for i, v in enumerate(stage_means['payload_mean']):
    ax3.text(i, v + 15, f'{v:.1f} B', ha='center', fontweight='bold')

# 4. TCP Window Mean
ax4 = axes[1, 1]
sns.barplot(x=stage_labels, y=stage_means['win_mean'], palette='Purples_r', ax=ax4)
ax4.set_title('Mean Advertised TCP Window Size (Bytes)', fontsize=13, fontweight='bold')
ax4.set_ylabel('Window Size (Bytes)')
ax4.tick_params(axis='x', rotation=15)
for i, v in enumerate(stage_means['win_mean']):
    ax4.text(i, v + 150, f'{v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


## 4. Feature Correlation Heatmap
Understanding interactions between volume, TCP handshake flags, entropy, and stealth signals:


In [ ]:
corr_subset = [
    'packet_count', 'ip_byte_count', 'packet_rate', 'byte_rate',
    'tcp_ratio', 'port_entropy', 'pure_syn_count', 'ack_count',
    'win_mean', 'payload_mean', 'auth_packet_ratio', 'mitre_stage_code'
]

corr_matrix = df[corr_subset].sample(n=50000, random_state=42).corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1.0, vmax=1.0, linewidths=0.5)
plt.title('Feature Correlation Matrix (50,000 Sample Slice)', fontsize=15, fontweight='bold')
plt.show()


## 5. Temporal Rolling Window Simulation & Attack Transition Timeline
Demonstrating how continuous 47-D State Vectors evolve over a sequence timeline during an actual attack transition (Benign $\to$ Denial of Service):


In [ ]:
# Extract an active sequence transition from DARPA Wednesday around attack onset (SYN Flood)
df_wed = df[df['day'] == 'wednesday'].reset_index(drop=True)
attack_indices = df_wed[df_wed['mitre_stage_code'] == 4].index

# Slice 50 windows before attack and 50 windows during attack
start_idx = max(0, attack_indices[0] - 50)
end_idx = min(len(df_wed), attack_indices[0] + 50)
session_sample = df_wed.iloc[start_idx:end_idx].reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# 1. Telemetry Plot: Packet Rate & Byte Rate
ax1.plot(session_sample.index, session_sample['packet_rate'], color='#e7298a', lw=2.5, label='Packet Rate (pkts/s)')
ax1.plot(session_sample.index, session_sample['byte_rate'] / 100.0, color='#2b5c8f', lw=2, linestyle='--', label='Byte Rate / 100 (B/s)')
ax1.set_ylabel('Telemetry Intensity', fontsize=12)
ax1.set_title('Real-Time State Telemetry Evolution (Attack Transition Timeline)', fontsize=14, fontweight='bold')
ax1.axvline(x=50, color='red', linestyle=':', lw=2, label='Attack Onset (t_0)')
ax1.legend(loc='upper left', fontsize=11)

# 2. Attack Stage Ground-Truth Transition
ax2.step(session_sample.index, session_sample['mitre_stage_code'], color='#d95f02', lw=3, where='post')
ax2.axvline(x=50, color='red', linestyle=':', lw=2)
ax2.set_yticks([0, 1, 2, 3, 4])
ax2.set_yticklabels(['0: Benign', '1: Recon', '2: Access', '3: Lateral', '4: DoS'])
ax2.set_xlabel('Rolling Sequence Window Index (t = 10s per window)', fontsize=12)
ax2.set_ylabel('MITRE Attack Stage', fontsize=12)
ax2.set_title('Ground-Truth Attack Stage Step Function', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
